In [0]:
dbutils.widgets.text("catalog","")
dbutils.widgets.text("schema","")

In [0]:
catalog=dbutils.widgets.get("catalog")
schema=dbutils.widgets.get("schema")

In [0]:
print(catalog)
print(schema)

In [0]:
spark.sql(f"USE CATALOG {catalog}")

In [0]:
spark.sql(f""" CREATE TABLE IF NOT EXISTS {catalog}.governance.tables (
    table_id            INT,
    table_name          STRING,
    source_system       STRING,        -- sqlserver / blob
    source_schema       STRING,        -- dbo (null for blob)
    source_table        STRING,        -- table name (null for blob)
    source_path         STRING,        -- blob path (null for sqlserver)
    target_layer        STRING,        -- cleansed/curated
    raw_schema       STRING,        -- raw
    cleansed_schema       STRING,        -- cleansed
    curated_schema         STRING,        -- curated
    active_flag         BOOLEAN,
    load_order          INT,
    created_at          TIMESTAMP
)
USING DELTA
""")

In [0]:
spark.sql(f""" CREATE TABLE IF NOT EXISTS {catalog}.governance.table_parameters (
    table_id            INT,
    parameter_name      STRING,        -- load_type / primary_key / watermark_column
    parameter_value     STRING,
    created_at          TIMESTAMP
)
USING DELTA""")

In [0]:
spark.sql(f"""CREATE TABLE IF NOT EXISTS {catalog}.governance.table_watermarks (
    table_id                INT,
    last_watermark_value    STRING,     -- flexible type storage
    last_updated_at         TIMESTAMP,
    last_run_id             BIGINT
)
USING DELTA
PARTITIONED BY (table_id); """)

In [0]:
spark.sql(f""" CREATE TABLE IF NOT EXISTS {catalog}.governance.pipeline_runs (
    run_id              BIGINT,
    table_id            INT,
    layer               STRING,        -- raw / cleansed / curated
    start_time          TIMESTAMP,
    end_time            TIMESTAMP,
    status              STRING,        -- SUCCESS / FAILED
    number_of_records     BIGINT,
    error_message       STRING
)
USING DELTA
PARTITIONED BY (table_id)""")

In [0]:
spark.sql(f""" CREATE SCHEMA IF NOT EXISTS {catalog}.source """)

In [0]:
spark.sql(f""" CREATE VOLUME IF NOT EXISTS {catalog}.source.volume""")

In [0]:
spark.sql(f""" 
INSERT INTO {catalog}.governance.tables VALUES
-- 1. Customers (SQL Server)
(1, 'customers', 'sqlserver', '{catalog}', 'customers', NULL, 'cleansed', 'raw', 'cleansed', NULL, TRUE, 1, current_timestamp()),

-- 2. Accounts (SQL Server)
(2, 'accounts', 'sqlserver', '{catalog}', 'accounts', NULL, 'cleansed', 'raw', 'cleansed', NULL, TRUE, 2, current_timestamp()),

-- 3. Transactions (SQL Server)
(3, 'transactions', 'sqlserver', '{catalog}', 'transactions', NULL, 'cleansed', 'raw', 'cleansed', NULL, TRUE, 3, current_timestamp()),

-- 4. Branches (SQL Server - Full Load)
(4, 'branches', 'sqlserver', '{catalog}', 'branches', NULL, 'cleansed', 'raw', 'cleansed', NULL, TRUE, 4, current_timestamp()),

-- 5. Credit Bureau Reports (Blob CSV)
(5, 'credit_bureau_reports', 'blob', NULL, NULL,
 '/Volumes/{catalog}/source/volume/credit_bureau_reports/', 'cleansed',
 'raw', 'cleansed', NULL, TRUE, 5, current_timestamp()),

-- 6. Payment Gateway Logs (Blob CSV)
(6, 'payment_gateway_logs', 'blob', NULL, NULL,
 '/Volumes/{catalog}/source/volume/payment_gateway_logs/', 'cleansed',
 'raw', 'cleansed', NULL, TRUE, 6, current_timestamp())""")

In [0]:
spark.sql(f"""
INSERT INTO {catalog}.governance.table_parameters VALUES

-- ================= CUSTOMERS =================
(1, 'load_type', 'MERGE', current_timestamp()),
(1, 'primary_key', 'customer_id', current_timestamp()),
(1, 'watermark_column', 'updated_at', current_timestamp()),

-- ================= ACCOUNTS =================
(2, 'load_type', 'MERGE', current_timestamp()),
(2, 'primary_key', 'account_id', current_timestamp()),
(2, 'watermark_column', 'updated_at', current_timestamp()),

-- ================= TRANSACTIONS =================
(3, 'load_type', 'APPEND', current_timestamp()),
(3, 'primary_key', 'txn_id', current_timestamp()),
(3, 'watermark_column', 'txn_timestamp', current_timestamp()),

-- ================= BRANCHES =================
(4, 'load_type', 'FULL', current_timestamp()),
(4, 'primary_key', 'branch_code', current_timestamp()),

-- ================= CREDIT BUREAU REPORTS =================
(5, 'load_type', 'MERGE', current_timestamp()),
(5, 'primary_key', 'customer_id', current_timestamp()),
(5, 'watermark_column', 'bureau_pull_date', current_timestamp()),

-- ================= PAYMENT GATEWAY LOGS =================
(6, 'load_type', 'APPEND', current_timestamp()),
(6, 'primary_key', 'txn_id', current_timestamp()),
(6, 'watermark_column', 'processed_timestamp', current_timestamp())""")



spark.sql(f"""INSERT INTO {catalog}.governance.table_watermarks VALUES
(1, '1900-01-01 00:00:00', current_timestamp(), NULL),
(2, '1900-01-01 00:00:00', current_timestamp(), NULL),
(3, '1900-01-01 00:00:00', current_timestamp(), NULL),
(5, '1900-01-01 00:00:00', current_timestamp(), NULL),
(6, '1900-01-01 00:00:00', current_timestamp(), NULL);
""")

spark.sql(f"""
INSERT INTO {catalog}.governance.tables
VALUES (
    7,
    'customer_360',
    'cleansed',
    NULL,
    NULL,
    NULL,
    'curated',
    NULL,
    NULL,
    'curated',
    TRUE,
    1,
    current_timestamp()
)""")



spark.sql(f"""
INSERT INTO {catalog}.governance.tables
VALUES
(
8,
'branch_performance',
'cleansed',
NULL,
NULL,
NULL,
'curated',
NULL,
NULL,
'curated',
TRUE,
2,
current_timestamp()
),

(
9,
'transaction_channel_summary',
'cleansed',
NULL,
NULL,
NULL,
'curated',
NULL,
NULL,
'curated',
TRUE,
3,
current_timestamp()
),

(
10,
'daily_bank_kpi',
'cleansed',
NULL,
NULL,
NULL,
'curated',
NULL,
NULL,
'curated',
TRUE,
4,
current_timestamp()
)""")



spark.sql(f"""INSERT INTO {catalog}.governance.tables
VALUES (
    11,
    'risk_customer_summary',
    'cleansed',
    NULL,
    NULL,
    NULL,
    'curated',
    NULL,
    NULL,
    'curated',
    TRUE,
    1,
    current_timestamp()
)""")